<a href="https://colab.research.google.com/github/Parakkrama24/cudaProgramming/blob/main/Cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def vecAdd(A, B, C):
    """
    CUDA kernel to perform element-wise vector addition: C = A + B
    """
    # Calculate the global index for the current thread
    idx = cuda.grid(1)

    # Ensure the index is within the bounds of the arrays
    if idx < A.size:
        C[idx] = A[idx] + B[idx]

# --- Example of how to use the kernel ---
# This part replaces the C++ main function and kernel invocation logic.
# Make sure you have a GPU runtime enabled in Colab to run this.

# Define array size
size = 256 * 10 # Using a larger size to better demonstrate GPU usage

# Create host arrays
A_host = np.random.rand(size).astype(np.float32)
B_host = np.random.rand(size).astype(np.float32)
C_host = np.zeros_like(A_host) # Result array

print("Input A (first 5 elements):", A_host[:5])
print("Input B (first 5 elements):", B_host[:5])

# Copy host arrays to device (GPU)
A_device = cuda.to_device(A_host)
B_device = cuda.to_device(B_host)
C_device = cuda.to_device(C_host)

# Configure the kernel launch: threads per block and blocks per grid
threads_per_block = 256 # A common choice for GPU threads
blocks_per_grid = (size + (threads_per_block - 1)) // threads_per_block

# Launch the kernel
print(f"Launching kernel with {blocks_per_grid} blocks and {threads_per_block} threads per block for size {size}...")
vecAdd[blocks_per_grid, threads_per_block](A_device, B_device, C_device)
cuda.synchronize() # Wait for the GPU to finish computations

# Copy the result back from device to host
C_host_result = C_device.copy_to_host()

print("Result C (first 5 elements):", C_host_result[:5])
print("Verification (A[0]+B[0]):", A_host[0] + B_host[0])
print("Is result correct (first element)?", np.isclose(C_host_result[0], A_host[0] + B_host[0]))


Input A (first 5 elements): [0.919083   0.289494   0.45908552 0.5077975  0.23234987]
Input B (first 5 elements): [0.8287948  0.98963785 0.9572361  0.30716035 0.54525834]
Launching kernel with 10 blocks and 256 threads per block for size 2560...
Result C (first 5 elements): [1.7478778  1.2791319  1.4163216  0.81495786 0.7776082 ]
Verification (A[0]+B[0]): 1.7478778
Is result correct (first element)? True


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 10 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
%%writefile vecadd_cuda.cu
#include <iostream>
#include <cuda_runtime.h> // For cudaMalloc, cudaMemcpy, cudaFree, cudaDeviceSynchronize
#include <stdlib.h>     // For rand, srand
#include <time.h>       // For time
#include <cmath>        // For fabs

// CUDA kernel to perform element-wise vector addition: C = A + B
__global__ void vecAdd(float* A, float* B, float* C, int size)
{
    // Calculate the global index for the current thread
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Ensure the index is within the bounds of the arrays
    if (idx < size)
    {
        C[idx] = A[idx] + B[idx];
    }
}

int main()
{
    srand(time(NULL)); // Seed the random number generator

    // Define array size
    int size = 256 * 10; // Using a larger size to better demonstrate GPU usage

    // Allocate host memory
    float *h_A, *h_B, *h_C;
    h_A = new float[size];
    h_B = new float[size];
    h_C = new float[size];

    // Initialize host arrays
    for (int i = 0; i < size; ++i)
    {
        h_A[i] = (float)rand() / RAND_MAX;
        h_B[i] = (float)rand() / RAND_MAX;
    }

    std::cout << "Input A (first 5 elements): ";
    for (int i = 0; i < 5; ++i) std::cout << h_A[i] << " ";
    std::cout << std::endl;

    std::cout << "Input B (first 5 elements): ";
    for (int i = 0; i < 5; ++i) std::cout << h_B[i] << " ";
    std::cout << std::endl;

    // Allocate device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size * sizeof(float));
    cudaMalloc(&d_B, size * sizeof(float));
    cudaMalloc(&d_C, size * sizeof(float));

    // Copy host arrays to device
    cudaMemcpy(d_A, h_A, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size * sizeof(float), cudaMemcpyHostToDevice);

    // Configure the kernel launch: threads per block and blocks per grid
    int threadsPerBlock = 256;
    int blocksPerGrid = (size + threadsPerBlock - 1) / threadsPerBlock;

    std::cout << "Launching kernel with " << blocksPerGrid << " blocks and "
              << threadsPerBlock << " threads per block for size " << size << "..." << std::endl;

    // Launch the kernel
    vecAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, size);

    // Wait for the GPU to finish computations
    cudaDeviceSynchronize();

    // Copy the result back from device to host
    cudaMemcpy(h_C, d_C, size * sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "Result C (first 5 elements): ";
    for (int i = 0; i < 5; ++i) std::cout << h_C[i] << " ";
    std::cout << std::endl;

    std::cout << "Verification (A[0]+B[0]): " << h_A[0] + h_B[0] << std::endl;

    // Due to floating point precision, direct equality might fail. Use an epsilon comparison.
    bool is_correct = fabs(h_C[0] - (h_A[0] + h_B[0])) < 1e-6;
    std::cout << "Is result correct (first element)? " << (is_correct ? "True" : "False") << std::endl;

    // Free device memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free host memory
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    return 0;
}

Overwriting vecadd_cuda.cu


In [ ]:
!nvcc vecadd_cuda.cu -o vecadd_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vecadd_cuda

Input A (first 5 elements): 0.615722 0.511625 0.433776 0.24776 0.129373 
Input B (first 5 elements): 0.105099 0.984676 0.312566 0.615266 0.938655 
Launching kernel with 10 blocks and 256 threads per block for size 2560...
Result C (first 5 elements): 0.720821 1.4963 0.746342 0.863026 1.06803 
Verification (A[0]+B[0]): 0.720821
Is result correct (first element)? True


In [ ]:
%%writefile scratch.cu

#include<stdio.h>
#define N 256 //totol elements

__global__ void vectorAdd(float* A, float *B ,float *C){

int workIndex = threadIdx.x + blockIdx.x* blockDim.x;
   if (workIndex < N) {
        C[workIndex] = A[workIndex] + B[workIndex];
       // printf("Thread %d -> C[%d] = %f\n", threadIdx.x, workIndex, C[workIndex]);
    }


}

int main(){


float A[N],B[N],C[N];

//Initialize values

for(int i=0;i<N;i++){
  A[i]=2.0f;
  B[i]=3.0f;
}

float *d_A,*d_B,*d_C;


  //Allocate memory on GPU
  cudaMalloc((void**)&d_A,N*sizeof(float));
  cudaMalloc((void**)&d_B,N*sizeof(float));
  cudaMalloc((void**)&d_C,N*sizeof(float));

  //copy data from cpu to GPU
  cudaMemcpy(d_A,A,N*sizeof(float),cudaMemcpyHostToDevice);
  cudaMemcpy(d_B,B,N*sizeof(float),cudaMemcpyHostToDevice);

// Launch kernel
    int threadsPerBlock = 16;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
    printf("threadsPerBlock %d \n blocksPerGrid %d \n",threadsPerBlock,blocksPerGrid);


    //int BlocksPerGrid= cuda::ceil_div(threadsPerBlock,blocksPerGrid);
    //printf("BlocksPerGrid %d \n",BlocksPerGrid);


    vectorAdd<<<blocksPerGrid,threadsPerBlock>>>(d_A,d_B,d_C);

    //copy back to cpu memory
    cudaMemcpy(C,d_C,N*sizeof(float),cudaMemcpyDeviceToHost);

  for(int i=0;i<N;i++){

    printf("C[%d] = %f\n",i,C[i]);
  }



  cudaDeviceSynchronize(); // Add this line to wait for the kernel to finish and print

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

  return 0;
}

Overwriting scratch.cu


In [ ]:
!nvcc scratch.cu -o scratch

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./scratch

threadsPerBlock 16 
 blocksPerGrid 16 
C[0] = 5.000000
C[1] = 5.000000
C[2] = 5.000000
C[3] = 5.000000
C[4] = 5.000000
C[5] = 5.000000
C[6] = 5.000000
C[7] = 5.000000
C[8] = 5.000000
C[9] = 5.000000
C[10] = 5.000000
C[11] = 5.000000
C[12] = 5.000000
C[13] = 5.000000
C[14] = 5.000000
C[15] = 5.000000
C[16] = 5.000000
C[17] = 5.000000
C[18] = 5.000000
C[19] = 5.000000
C[20] = 5.000000
C[21] = 5.000000
C[22] = 5.000000
C[23] = 5.000000
C[24] = 5.000000
C[25] = 5.000000
C[26] = 5.000000
C[27] = 5.000000
C[28] = 5.000000
C[29] = 5.000000
C[30] = 5.000000
C[31] = 5.000000
C[32] = 5.000000
C[33] = 5.000000
C[34] = 5.000000
C[35] = 5.000000
C[36] = 5.000000
C[37] = 5.000000
C[38] = 5.000000
C[39] = 5.000000
C[40] = 5.000000
C[41] = 5.000000
C[42] = 5.000000
C[43] = 5.000000
C[44] = 5.000000
C[45] = 5.000000
C[46] = 5.000000
C[47] = 5.000000
C[48] = 5.000000
C[49] = 5.000000
C[50] = 5.000000
C[51] = 5.000000
C[52] = 5.000000
C[53] = 5.000000
C[54] = 5.000000
C[55] = 5.000000
C[56] = 5.000000
C[

In [ ]:
%%writefile unifiedMemo.cu
#include <stdio.h>

#define N 256

__global__ void vecAdd(float* A, float* B, float* C) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    float *A, *B, *C;

    // Unified Memory allocation
    cudaMallocManaged(&A, N * sizeof(float));
    cudaMallocManaged(&B, N * sizeof(float));
    cudaMallocManaged(&C, N * sizeof(float));

    // Initialize on CPU
    for (int i = 0; i < N; i++) {
        A[i] = 2.0f;
        B[i] = 3.0f;
    }

    // Launch kernel
    vecAdd<<<1, N>>>(A, B, C);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Use result directly (no memcpy!)
    for (int i = 0; i < 5; i++) {
        printf("C[%d] = %f\n", i, C[i]);
    }

    cudaFree(A);
    cudaFree(B);
    cudaFree(C);

    return 0;
}

Overwriting unifiedMemo.cu


In [ ]:
!nvcc unifiedMemo.cu -o unifiedMemo

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./unifiedMemo

C[0] = 5.000000
C[1] = 5.000000
C[2] = 5.000000
C[3] = 5.000000
C[4] = 5.000000


In [ ]:
%%writefile explicitMemo.cu


int main(){

int vectorLength=10;

//Poniters for host memeory
float *A = nullptr;
float *B = nullptr;
float *C = nullptr;
float* comparisonResult = (float*)malloc(vectorLength*sizeof(float));

//Pointers for device memory
float *d_A = nullptr;
float *d_B = nullptr;
float *d_C = nullptr;

//Allocate Host Memeory using cudaMallocHost API. This is best practice
//When buffers will be used for copis between cpu and gpus Memory
cudaMallocHost(&A, N * sizeof(float));
cudaMallocHost(&B, N * sizeof(float));
cudaMallocHost(&C, N * sizeof(float));

//Initialize vectores on the host
initArray(A,vectorLength);
initArray(B,vectorLength);

//Allocate device memory
cudaMalloc(&d_A,vectorLength*sizeof(float));
cudaMalloc(&d_B,vectorLength*sizeof(float));
cudaMalloc(&d_C,vectorLength*sizeof(float));

//Copy data from host to device`
cudaMemcpy(d_A,A,vectorLength*sizeof(float),cudaMemcpyDefault);
cudaMemcpy(d_B,B,vectorLength*sizeof(float),cudaMemcpyDefault);
//set all values to 0
cudaMemset(d_C,0,vectorLength*sizeof(float));

//Launch kernel
int threadsPerBlock = 256;
int blocksPerGrid = (vectorLength + threadsPerBlock - 1) / threadsPerBlock
vecAdd<<<blocksPerGrid,threadsPerBlock>>>(d_A,d_B,d_C);

//wait for GPU to finish
cudaDeviceSynchronize();

cudayMemcpy(C,d_C,vectorLength*sizeof(float),cudaMemcpyDefault);


  // Perform computation serially on CPU for comparison
    serialVecAdd(A, B, comparisonResult, vectorLength);

    // Confirm that CPU and GPU got the same answer
    if(vectorApproximatelyEqual(C, comparisonResult, vectorLength))
    {
        printf("Explicit Memory: CPU and GPU answers match\n");
    }
    else
    {
        printf("Explicit Memory: Error - CPU and GPU answers to not match\n");
    }

    // clean up
    cudaFree(devA);
    cudaFree(devB);
    cudaFree(devC);
    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);
    free(comparisonResult);
}



In [ ]:
%%writefile bothTogetherOne.cu

#include <cuda_runtime_api.h>
#include <memory.h>
#include <cstdlib>
#include <ctime>
#include <stdio.h>
#include <cuda/cmath>

__global__ void vecAdd(float* A, float* B, float* C, int vectorLength)
{
    int workIndex = threadIdx.x + blockIdx.x*blockDim.x;
    if(workIndex < vectorLength)
    {
        C[workIndex] = A[workIndex] + B[workIndex];
    }
}

void initArray(float* A, int length)
{
     std::srand(std::time({}));
    for(int i=0; i<length; i++)
    {
        A[i] = rand() / (float)RAND_MAX;
    }
}

void serialVecAdd(float* A, float* B, float* C,  int length)
{
    for(int i=0; i<length; i++)
    {
        C[i] = A[i] + B[i];
    }
}

bool vectorApproximatelyEqual(float* A, float* B, int length, float epsilon=0.00001)
{
    for(int i=0; i<length; i++)
    {
        if(fabs(A[i] -B[i]) > epsilon)
        {
            printf("Index %d mismatch: %f != %f", i, A[i], B[i]);
            return false;
        }
    }
    return true;
}

//unified-memory-begin
void unifiedMemExample(int vectorLength)
{
    // Pointers to memory vectors
    float* A = nullptr;
    float* B = nullptr;
    float* C = nullptr;
    float* comparisonResult = (float*)malloc(vectorLength*sizeof(float));

    // Use unified memory to allocate buffers
    cudaMallocManaged(&A, vectorLength*sizeof(float));
    cudaMallocManaged(&B, vectorLength*sizeof(float));
    cudaMallocManaged(&C, vectorLength*sizeof(float));

    // Initialize vectors on the host
    initArray(A, vectorLength);
    initArray(B, vectorLength);

    // Launch the kernel. Unified memory will make sure A, B, and C are
    // accessible to the GPU
    int threads = 256;
    int blocks = cuda::ceil_div(vectorLength, threads);
    vecAdd<<<blocks, threads>>>(A, B, C, vectorLength);
    // Wait for the kernel to complete execution
    cudaDeviceSynchronize();

    // Perform computation serially on CPU for comparison
    serialVecAdd(A, B, comparisonResult, vectorLength);

    // Confirm that CPU and GPU got the same answer
    if(vectorApproximatelyEqual(C, comparisonResult, vectorLength))
    {
        printf("Unified Memory: CPU and GPU answers match\n");
    }
    else
    {
        printf("Unified Memory: Error - CPU and GPU answers do not match\n");
    }

    // Clean Up
    cudaFree(A);
    cudaFree(B);
    cudaFree(C);
    free(comparisonResult);

}
//unified-memory-end


int main(int argc, char** argv)
{
    int vectorLength = 1024;
    if(argc >=2)
    {
        vectorLength = std::atoi(argv[1]);
    }
    unifiedMemExample(vectorLength);
    return 0;
}

Writing bothTogetherOne.cu


In [ ]:
!nvcc bothTogetherOne.cu -o bothTogetherOne

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./bothTogetherOne

Unified Memory: CPU and GPU answers match


In [ ]:
#include <cuda_runtime_api.h>
#include <memory.h>
#include <cstdlib>
#include <ctime>
#include <stdio.h>
#include <cuda/cmath>

__global__ void vecAdd(float* A, float* B, float* C, int vectorLength)
{
    int workIndex = threadIdx.x + blockIdx.x*blockDim.x;
    if(workIndex < vectorLength)
    {
        C[workIndex] = A[workIndex] + B[workIndex];
    }
}

void initArray(float* A, int length)
{
     std::srand(std::time({}));
    for(int i=0; i<length; i++)
    {
        A[i] = rand() / (float)RAND_MAX;
    }
}

void serialVecAdd(float* A, float* B, float* C,  int length)
{
    for(int i=0; i<length; i++)
    {
        C[i] = A[i] + B[i];
    }
}

bool vectorApproximatelyEqual(float* A, float* B, int length, float epsilon=0.00001)
{
    for(int i=0; i<length; i++)
    {
        if(fabs(A[i] -B[i]) > epsilon)
        {
            printf("Index %d mismatch: %f != %f", i, A[i], B[i]);
            return false;
        }
    }
    return true;
}

//explicit-memory-begin
void explicitMemExample(int vectorLength)
{
    // Pointers for host memory
    float* A = nullptr;
    float* B = nullptr;
    float* C = nullptr;
    float* comparisonResult = (float*)malloc(vectorLength*sizeof(float));

    // Pointers for device memory
    float* devA = nullptr;
    float* devB = nullptr;
    float* devC = nullptr;

    //Allocate Host Memory using cudaMallocHost API. This is best practice
    // when buffers will be used for copies between CPU and GPU memory
    cudaMallocHost(&A, vectorLength*sizeof(float));
    cudaMallocHost(&B, vectorLength*sizeof(float));
    cudaMallocHost(&C, vectorLength*sizeof(float));

    // Initialize vectors on the host
    initArray(A, vectorLength);
    initArray(B, vectorLength);

    // start-allocate-and-copy
    // Allocate memory on the GPU
    cudaMalloc(&devA, vectorLength*sizeof(float));
    cudaMalloc(&devB, vectorLength*sizeof(float));
    cudaMalloc(&devC, vectorLength*sizeof(float));

    // Copy data to the GPU
    cudaMemcpy(devA, A, vectorLength*sizeof(float), cudaMemcpyDefault);
    cudaMemcpy(devB, B, vectorLength*sizeof(float), cudaMemcpyDefault);
    cudaMemset(devC, 0, vectorLength*sizeof(float));
    // end-allocate-and-copy

    // Launch the kernel
    int threads = 256;
    int blocks = cuda::ceil_div(vectorLength, threads);
    vecAdd<<<blocks, threads>>>(devA, devB, devC, vectorLength);
    // wait for kernel execution to complete
    cudaDeviceSynchronize();

    // Copy results back to host
    cudaMemcpy(C, devC, vectorLength*sizeof(float), cudaMemcpyDefault);

    // Perform computation serially on CPU for comparison
    serialVecAdd(A, B, comparisonResult, vectorLength);

    // Confirm that CPU and GPU got the same answer
    if(vectorApproximatelyEqual(C, comparisonResult, vectorLength))
    {
        printf("Explicit Memory: CPU and GPU answers match\n");
    }
    else
    {
        printf("Explicit Memory: Error - CPU and GPU answers to not match\n");
    }

    // clean up
    cudaFree(devA);
    cudaFree(devB);
    cudaFree(devC);
    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);
    free(comparisonResult);
}
//explicit-memory-end


int main(int argc, char** argv)
{
    int vectorLength = 1024;
    if(argc >=2)
    {
        vectorLength = std::atoi(argv[1]);
    }
    explicitMemExample(vectorLength);
    return 0;
}